# NB01 · Part 1 · Essentials (pages 5–18)
Each **Page N** cell is copied exactly from the book. Cells starting with `# TEST SETUP` are not in the book; they create the inputs the next card needs.

**To test:** attach the same test lakehouse, then *Run all*. If a cell fails, note the page number, then use *Run cells below* to carry on.

Cells that end or restart the session are in **NB03**, so they don't stop *Run all*.

In [ ]:
%run NB00_Setup_Sample_Data

## Page 6 · OneLake

**Page 6 · Read a shortcut like any table**  
`crm_customers` is a normal table here, standing in for a shortcut.

In [ ]:
# shortcut named crm_customers in Tables/
df = spark.read.table("crm_customers")

## Page 7 · Tables and Files

**Page 7 · Promote a file to a table**

In [ ]:
raw = (spark.read.option("header", True)
             .csv("Files/raw/orders.csv"))

(raw.write.mode("overwrite")
    .saveAsTable("bronze_orders"))

**Page 7 · Is this folder a Delta table?**

In [ ]:
from delta.tables import DeltaTable

DeltaTable.isDeltaTable(
    spark, "Tables/silver_sales")

## Page 9 · Medallion rules

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
spark.sql("DROP TABLE IF EXISTS bronze_orders")  # a fresh bronze table for this example

**Page 9 · Bronze: append with load metadata**

In [ ]:
raw = (spark.read.option("header", True)
         .csv("Files/landing/orders/"))

(raw.withColumn("_ingested_at",
                F.current_timestamp())
    .withColumn("_source_file",
                F.col("_metadata.file_path"))
    .write.mode("append")
    .saveAsTable("bronze_orders"))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

# an empty silver table with the cleaned schema, so MERGE has a target
_clean_schema = (spark.read.table("bronze_orders")
                 .withColumn("amount", F.col("amount").cast("decimal(18,2)")).limit(0))
_clean_schema.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_orders")

**Page 9 · Silver: clean, dedupe, upsert**

In [ ]:
from pyspark.sql import Window as W
from delta.tables import DeltaTable

w = (W.partitionBy("order_id")
      .orderBy(F.desc("_ingested_at")))

clean = (spark.read.table("bronze_orders")
  .withColumn("amount",
       F.col("amount").cast("decimal(18,2)"))
  .withColumn("rn", F.row_number().over(w))
  .filter("rn = 1").drop("rn"))

(DeltaTable.forName(spark, "silver_orders")
   .alias("t")
   .merge(clean.alias("s"),
          "t.order_id = s.order_id")
   .whenMatchedUpdateAll()
   .whenNotMatchedInsertAll()
   .execute())

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
spark.sql("DROP TABLE IF EXISTS gold_revenue")  # page 9 and page 17 use different gold shapes

**Page 9 · Gold: rebuild a summary**

In [ ]:
(spark.read.table("silver_orders")
   .groupBy("order_month", "country")
   .agg(F.sum("amount").alias("revenue"))
   .write.mode("overwrite")
   .saveAsTable("gold_revenue"))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
spark.sql("SELECT * FROM gold_revenue ORDER BY revenue DESC").show(5)

## Page 10 · Notebook essentials

**Page 10 · notebookutils: files**

In [ ]:
fs = notebookutils.fs

fs.ls("Files/raw")                 # list
fs.mkdirs("Files/archive/2026")    # new folder
fs.cp("Files/raw/a.csv",
      "Files/archive/2026/a.csv")  # copy
fs.mv("Files/raw/b.csv",
      "Files/archive/2026/b.csv")  # move
fs.rm("Files/tmp", True)           # delete all

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
display(notebookutils.fs.ls("Files/archive/2026"))  # expect a.csv and b.csv

**Page 10 · notebookutils: run, stop, context**  
Needs the small notebook **nb_silver** (included) in the same workspace. `nb.exit` and `session.stop()` from this card are in NB03.

In [ ]:
nb = notebookutils.notebook
nb.run("nb_silver", 1800,
       {"run_date": "2026-10-19"})

**Page 10 · notebookutils: run, stop, context**

In [ ]:
notebookutils.runtime.context # run info
help(notebookutils)           # everything else

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 10 · See your data**

In [ ]:
display(df.limit(100))   # grid, charts, profile
df.printSchema()
df.summary().show()      # count, mean, min, max

## Page 11 · Read data

The `abfss://…` path on page 11 is an address pattern, not code: copy a real one from the lakehouse explorer (… › Copy ABFS path) and check it with `spark.read.csv(path, header=True).show(3)`.

**Page 11 · CSV with a header row**

In [ ]:
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("Files/raw/sales.csv"))

**Page 11 · Parquet and JSON**

In [ ]:
# a folder reads every file inside it
df = spark.read.parquet("Files/raw/sales/")

df = (spark.read
      .option("multiLine", True)
      .json("Files/raw/events.json"))

**Page 11 · Delta tables**

In [ ]:
# by name (default lakehouse)
df = spark.read.table("sales")

# by path
df = (spark.read.format("delta")
        .load("Tables/sales"))

# with SQL
df = spark.sql(
    "SELECT * FROM sales WHERE year = 2026")

**Page 11 · Explicit schema for scheduled jobs**

In [ ]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("order_id", T.StringType()),
    T.StructField("order_date", T.DateType()),
    T.StructField("amount", T.DoubleType()),
])

df = (spark.read.schema(schema)
      .option("header", True)
      .csv("Files/raw/sales.csv"))

**Page 11 · Look before you load**

In [ ]:
notebookutils.fs.ls("Files/raw") # list files
display(df.limit(20))            # rich grid
df.printSchema()                 # column types

## Page 12 · Write data

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 12 · Save as a Delta table**

In [ ]:
(df.write
   .mode("overwrite")
   .format("delta")
   .saveAsTable("silver_sales"))

# add new rows
(df.write.mode("append")
   .format("delta")
   .saveAsTable("silver_sales"))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales().withColumn("channel", F.lit("web"))  # one new column

**Page 12 · When the schema changes**

In [ ]:
# append, adding new columns
(df.write.mode("append")
   .option("mergeSchema", "true")
   .saveAsTable("silver_sales"))

# overwrite with a new schema
(df.write.mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("silver_sales"))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

df = sample_sales()
updates = (df.filter(F.col("country") == "UK").limit(50)
             .withColumn("amount", F.col("amount") + 1)
           .unionByName(df.limit(5).withColumn("order_id", F.concat(F.lit("NEW-"), "order_id"))))

**Page 12 · Upsert with MERGE**

In [ ]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark,
                            "silver_sales")

(target.alias("t")
   .merge(updates.alias("s"),
          "t.order_id = s.order_id")
   .whenMatchedUpdateAll()
   .whenNotMatchedInsertAll()
   .execute())

**Page 12 · Partition a large table**

In [ ]:
(df.write.mode("overwrite")
   .partitionBy("year")
   .saveAsTable("gold_sales_history"))

**Page 12 · Export files**

In [ ]:
(df.write.mode("overwrite")
   .parquet("Files/export/sales"))

# one CSV file, small outputs only
(df.coalesce(1).write.mode("overwrite")
   .option("header", True)
   .csv("Files/export/sales_csv"))

## Page 13 · Transform columns

**Page 13**

In [ ]:
# every snippet on pages 13–14 assumes these imports
from pyspark.sql import functions as F, types as T

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

df = spark.createDataFrame(
    [("O1", 120.0, "C001", "2026-10-19 10:00", "a.csv"), ("O2", 80.5, "C002", "2026-10-19 11:00", "b.csv")],
    "order_id string, amt double, cust string, load_ts string, source_file string")

**Page 13 · Pick, rename, drop**

In [ ]:
df = df.select("order_id",
               F.col("amt").alias("amount"))
df = df.withColumnRenamed("cust", "customer_id")
df = df.drop("load_ts", "source_file")

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 13 · Add and compute columns**

In [ ]:
df = (df
  .withColumn("vat", F.col("amount") * 0.2)
  .withColumn("source", F.lit("erp"))
  .withColumn("size",
     F.when(F.col("amount") >= 1000, "large")
      .when(F.col("amount") >= 100, "medium")
      .otherwise("small")))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales().withColumn("email", F.when(F.rand() > 0.1, F.lit("someone@example.com")))

**Page 13 · Filter rows**

In [ ]:
# wrap each condition in brackets
df = df.filter((F.col("amount") > 100) &
               (F.col("country") == "UK"))
df = df.filter(
    F.col("country").isin(["UK", "IE"]))
df = df.filter(F.col("email").isNotNull())

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 13 · Sort and remove duplicates**

In [ ]:
df = df.orderBy(F.desc("order_date"))
df = df.dropDuplicates(["order_id"])

**Page 13 · Cast types**

In [ ]:
df = df.withColumn("amount",
       F.col("amount").cast("decimal(18,2)"))
df = df.withColumn("qty",
       F.col("qty").cast(T.IntegerType()))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

df = (sample_sales()
      .withColumn("mobile", F.when(F.rand() > 0.5, F.lit("07700 900123")))
      .withColumn("landline", F.when(F.rand() > 0.5, F.lit("020 7946 0000"))))

**Page 13 · Handle nulls**

In [ ]:
df = df.fillna({"amount": 0,
                "country": "Unknown"})
df = df.na.drop(subset=["order_id"])
df = df.withColumn("phone",
       F.coalesce("mobile", "landline",
                  F.lit("n/a")))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = spark.createDataFrame([(1, "x")], "` Order ID ` int, `Customer Name ` string")

**Page 13 · Clean every column name at once**

In [ ]:
df = df.toDF(*[
    c.strip().lower().replace(" ", "_")
    for c in df.columns])

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df.printSchema()  # expect order_id, customer_name

## Page 14 · Dates, aggregates, JSON

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

df = spark.createDataFrame(
    [("  jane SMITH ", "+44 (0)20 7946-0000", "Jane Smith", "SKU-001", "Laptop")],
    "name string, phone string, full_name string, sku string, product string")

**Page 14 · Strings**

In [ ]:
df = (df
  .withColumn("name", F.trim(F.initcap("name")))
  .withColumn("phone",
       F.regexp_replace("phone", r"[^0-9]", ""))
  .withColumn("first",
       F.split("full_name", " ").getItem(0))
  .withColumn("label",
       F.concat_ws(" - ", "sku", "product"))
  .withColumn("code", F.substring("sku", 1, 3)))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

df = spark.createDataFrame(
    [("19/10/2026", "2026-10-19 09:30:00", "2026-10-23")],
    "order_date string, order_ts string, ship_date string")

**Page 14 · Dates and timestamps**

In [ ]:
df = (df
  .withColumn("order_date",
       F.to_date("order_date", "dd/MM/yyyy"))
  .withColumn("order_ts",
       F.to_timestamp("order_ts",
                      "yyyy-MM-dd HH:mm:ss"))
  .withColumn("year", F.year("order_date"))
  .withColumn("month_start",
       F.trunc("order_date", "month"))
  .withColumn("days_to_ship",
       F.datediff("ship_date", "order_date"))
  .withColumn("due",
       F.add_months("order_date", 1))
  .withColumn("loaded_on", F.current_date()))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 14 · Group and aggregate**

In [ ]:
gold = (df.groupBy("country", "year")
  .agg(F.sum("amount").alias("revenue"),
       F.countDistinct("customer_id")
        .alias("customers"),
       F.avg("amount").alias("avg_order"),
       F.max("order_date").alias("last_order")))

**Page 14 · Pivot**

In [ ]:
# listing the values saves an extra job
wide = (df.groupBy("product")
          .pivot("year", [2025, 2026])
          .sum("amount"))

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

payload_schema = T.StructType([
    T.StructField("customer", T.StringType()),
    T.StructField("items", T.ArrayType(T.StructType([
        T.StructField("sku", T.StringType()), T.StructField("qty", T.IntegerType())]))),
])
df = spark.createDataFrame(
    [("O1", '{"customer": "C001", "items": [{"sku": "SKU-001", "qty": 1}, {"sku": "SKU-004", "qty": 2}]}')],
    "order_id string, payload string")

**Page 14 · Nested JSON and arrays**

In [ ]:
# JSON string column to struct, then flatten
df = df.withColumn("data",
       F.from_json("payload", payload_schema))
df = df.select("order_id", "data.*")

# one row per array item
lines = df.select(
    "order_id",
    F.explode("items").alias("item"))

## Page 15 · Joins

**Page 15 · Three ways to say the key**

In [ ]:
# same column name: one key column in the result
orders.join(customers, "customer_id", "left")

# several columns
orders.join(customers,
            ["customer_id", "region"])

# different names: both columns stay
orders.join(customers,
            orders.cust_id == customers.id,
            "left")

**Page 15 · Aliases stop ambiguous columns**

In [ ]:
o = orders.alias("o")
c = customers.alias("c")

cond = F.col("o.cust_id") == F.col("c.id")
df = (o.join(c, cond, "left")
       .select("o.*",
         F.col("c.name").alias("customer")))

**Page 15 · Broadcast a small dimension**

In [ ]:
df = orders.join(F.broadcast(dim_country),
                 "country_code", "left")

**Page 15 · Find orphan rows**

In [ ]:
# orders whose customer is missing
orphans = orders.join(customers, "customer_id",
                      "left_anti")

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
a = orders.select("order_id", "region")
b = customers.select("customer_id", "region")

**Page 15 · Match null keys too**

In [ ]:
a.join(b, a.region.eqNullSafe(b.region))

## Page 16 · Window functions

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 16 · Latest row per key (dedupe)**

In [ ]:
from pyspark.sql import Window as W

w = (W.partitionBy("customer_id")
      .orderBy(F.desc("updated_at")))

latest = (df
  .withColumn("rn", F.row_number().over(w))
  .filter("rn = 1")
  .drop("rn"))

**Page 16 · Previous and next row**

In [ ]:
w = (W.partitionBy("customer_id")
       .orderBy("order_date"))

df = (df
  .withColumn("prev_amount",
       F.lag("amount", 1).over(w))
  .withColumn("next_date",
       F.lead("order_date", 1).over(w)))

**Page 16 · Running total and moving average**

In [ ]:
w = (W.partitionBy("customer_id")
       .orderBy("order_date"))

running = w.rowsBetween(W.unboundedPreceding,
                        W.currentRow)
last_7 = w.rowsBetween(-6, 0)

df = (df
  .withColumn("running",
       F.sum("amount").over(running))
  .withColumn("avg_7",
       F.avg("amount").over(last_7)))

**Page 16 · Share of the group total**

In [ ]:
by_cust = W.partitionBy("customer_id")

df = df.withColumn("share",
       F.col("amount")
       / F.sum("amount").over(by_cust))

## Page 17 · Spark SQL

**Page 17 · A SQL cell**

In [ ]:
%%sql
SELECT country,
       SUM(amount) AS revenue
FROM   silver_sales
GROUP  BY country
ORDER  BY revenue DESC

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
df = sample_sales()

**Page 17 · DataFrame to SQL and back**

In [ ]:
df.createOrReplaceTempView("v_orders")

big = spark.sql("""
    SELECT * FROM v_orders
    WHERE amount > 100
""")

**Page 17 · Safe parameters**

In [ ]:
df = spark.sql(
    "SELECT * FROM silver_sales "
    "WHERE year = :yr",
    args={"yr": 2026})

**Page 17 · Build a gold table**

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_revenue
USING DELTA AS
SELECT country, year, SUM(amount) AS revenue
FROM   silver_sales
GROUP  BY country, year

In [ ]:
# TEST SETUP (not in the book): inputs for the next card

(spark.read.table("silver_customers").limit(10)
      .withColumn("tier", F.lit("gold"))
 .unionByName(spark.createDataFrame([("C999", "new customer", "EU", "bronze")],
                                    "customer_id string, name string, region string, tier string"))
 .createOrReplaceTempView("v_updates"))

**Page 17 · MERGE in SQL**

In [ ]:
%%sql
MERGE INTO silver_customers AS t
USING v_updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

**Page 17 · Table history and time travel**  
Needs at least four versions of `silver_sales`; the cells above create them.

In [ ]:
%%sql
DESCRIBE HISTORY silver_sales;
DESCRIBE DETAIL  silver_sales;
SELECT * FROM silver_sales VERSION AS OF 3;

## Page 18 · Parameters and orchestration

**Page 18 · 1. Make a parameter cell**  
In Fabric, mark this cell as the parameter cell (… › Toggle parameter cell).

In [ ]:
# cell menu › Toggle parameter cell
run_date = "2026-10-19"
full_load = False

**Page 18 · 2. Call one notebook from another**  
The `exit` line from this card is the last cell of the included **nb_silver** notebook.

In [ ]:
result = notebookutils.notebook.run(
    "nb_silver",              # notebook name
    1800,                     # timeout, seconds
    {"run_date": run_date})

In [ ]:
# TEST SETUP (not in the book): inputs for the next card
print(result)  # rows written by nb_silver

**Page 18 · 3. Run independent notebooks in parallel**  
Needs the included **nb_dim_customer** and **nb_dim_product** notebooks.

In [ ]:
notebookutils.notebook.runMultiple(
    ["nb_dim_customer", "nb_dim_product"])